In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 05 · A model-chosen tool loop, and Mistral as the provider

In notebooks 01–03 the model writes text and the workflow fixes the order of steps. Here the model chooses the *next action*: a tool call or a final answer. That makes the model call a non-deterministic side effect (primer §1.3): ask again after a crash and it may choose a different call. The fix is to **journal the decision before acting on it**, then act at most once under a stable idempotency key. The second half swaps in Mistral as the model provider (offline, with a fake client) and shows the same loop on Mistral Workflows, where the platform provides the journal, the retries and the wait.

In [ ]:
from datetime import timedelta
import json
from lra import Engine, Event, Budget, Workflow, Next, Done, Wait, StepFailed, RunStatus, StepTask, SimulatedCrash
from lra.adapters.memory import FakeClock, FakeLLM, InMemoryStateStore, InMemoryTaskQueue, InMemoryEventBus, LocalRunner
from lra.examples import ALL_WORKFLOWS
from lra.examples.scripted import research_routes

def harness(routes=None, fail_times=0, chaos=None, lease_ttl_s=60):
    """One engine over in-memory adapters, driven the way Cloud Tasks would drive it."""
    clock, store, bus = FakeClock(), InMemoryStateStore(), InMemoryEventBus()
    queue = InMemoryTaskQueue(clock)
    llm = FakeLLM(routes=routes or research_routes(), fail_times=fail_times)
    engine = Engine(store=store, queue=queue, bus=bus, llm=llm, clock=clock, workflows=ALL_WORKFLOWS,
                    worker_id="worker-A", lease_ttl=timedelta(seconds=lease_ttl_s), chaos=chaos)
    return engine, LocalRunner(engine, queue, clock), clock, store, queue, bus

def show(run):
    print(f"{run.run_id} {run.status.value:12s} step={run.current_step} attempts={run.attempts} "
          f"wait={run.wait.key if run.wait else None} steps_used={run.budget.steps_used}")

In [ ]:
from lra.examples.tool_agent import PaymentAPI, ScriptedDecider, make_tool_agent

def tool_harness(wf, chaos=None):
    clock, store, bus = FakeClock(), InMemoryStateStore(), InMemoryEventBus()
    queue = InMemoryTaskQueue(clock)
    engine = Engine(store=store, queue=queue, bus=bus, llm=FakeLLM(), clock=clock, workflows=[wf],
                    worker_id="worker-A", lease_ttl=timedelta(seconds=60), chaos=chaos)
    return engine, LocalRunner(engine, queue, clock), clock, store

def show_journal(run):
    for e in run.state.get("journal", []):
        print("   ", e)

## Your turn: the two effect records of a turn

`decide` records the model's decision with `ctx.effect` *before* anything acts on it; `act` runs the tool at most once, passing an idempotency key that is stable across retries (`run_id:N`, N = the intent's position in the journal) and unique across runs. Fill in both keys.

In [ ]:
# TODO: fill in the blanks: decide_key, idem_key
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in the blanks: decide_key, idem_key")
def build_loop(decider, tools):
    wf = Workflow("my_loop", default_budget=Budget(max_steps=10))

    @wf.step(start=True)
    def decide(ctx):
        journal = ctx.state.setdefault("journal", [])
        n = len(journal)
        d = ctx.effect(____, lambda: decider.decide(ctx.input["goal"], journal))
        journal.append({"type": "decision", **d})
        if "final" in d:
            return Done(d["final"])
        journal.append({"type": "intent", "tool": d["tool"], "args": d["args"], "key": ____, "done": False})
        return Next("act")

    @wf.step()
    def act(ctx):
        journal = ctx.state["journal"]
        intent, n = journal[-1], len(journal) - 1
        intent["result"] = ctx.effect(f"act:{n}", lambda: tools[intent["tool"]](**intent["args"], key=intent["key"]))
        intent["done"] = True
        return Next("decide")

    return wf

# A model that would choose differently if asked again, and a crash right after the first decision.
crashed = []
def chaos(point, run):
    if point == "after_step_before_commit" and run.current_step == "decide" and not crashed:
        crashed.append(run.run_id); raise SimulatedCrash()

pay = PaymentAPI()
decider = ScriptedDecider([{"tool": "charge", "args": {"amount": 42}}, {"tool": "charge", "args": {"amount": 99}}, {"final": "done"}])
engine, runner, clock, store = tool_harness(build_loop(decider, {"charge": pay}), chaos=chaos)
run = engine.start("my_loop", {"goal": "pay invoice 42"})
try:
    runner.run_until_idle()
except SimulatedCrash:
    print("crashed after the model decided, before the checkpoint")
clock.advance(seconds=61); engine.reap(); runner.run_until_idle()
r = store.get(run.run_id); show(r); show_journal(r)
print("charges:", pay.charges, "| model calls:", decider.calls)
assert pay.charges == {f"{run.run_id}:1": 42.0, f"{run.run_id}:3": 99.0} and decider.calls == 3

Without the decision record, the retry would have re-asked the model and acted on a different answer (`99` instead of the `42` it had chosen): the world and the run would disagree about what the agent decided. With it, the model was asked three times in all: `42` (recorded once, reused by the retry), then `99`, then `final`.

## Crash after the charge

The library version, `lra.examples.tool_agent.make_tool_agent`, is the same loop plus an approval gate. Crash right after the charge: the retry re-executes *the same intent under the same key*, finds the effect record, and never re-asks the model.

In [ ]:
crashed = []
def chaos(point, run):
    if point == "after_step_before_commit" and run.current_step == "act" and not crashed:
        crashed.append(run.run_id); raise SimulatedCrash()

pay, decider = PaymentAPI(), ScriptedDecider([{"tool": "charge", "args": {"amount": 42}}, {"final": "Charged 42."}])
engine, runner, clock, store = tool_harness(make_tool_agent(decider, {"charge": pay}, gated=()), chaos=chaos)
run = engine.start("tool_agent", {"goal": "pay invoice 42"})
try:
    runner.run_until_idle()
except SimulatedCrash:
    print("crashed after the charge, before the checkpoint | charges so far:", pay.charges)
clock.advance(seconds=61); engine.reap(); runner.run_until_idle()
r = store.get(run.run_id); show(r)
print("charges:", pay.charges, "| payment API calls:", pay.calls, "| model calls:", decider.calls)
assert pay.calls == 1 and decider.calls == 2 and r.status == RunStatus.SUCCEEDED

## Approve what you execute

A gated tool parks the run with the *exact* proposed call in the journal. Approval executes that call; a rejection becomes the tool's result, so the model can choose again. A double click is a no-op.

In [ ]:
pay = PaymentAPI()
decider = ScriptedDecider([{"tool": "charge", "args": {"amount": 4200}}, {"final": "Not paid: vendor not onboarded."}])
engine, runner, clock, store = tool_harness(make_tool_agent(decider, {"charge": pay}))
run = engine.start("tool_agent", {"goal": "pay invoice 4200"})
runner.run_until_idle()
r = store.get(run.run_id); show(r); show_journal(r)
reject = Event(run_id=run.run_id, key=r.wait.key, payload={"decision": "reject", "by": "cfo", "comment": "vendor not onboarded"})
print("first:", engine.resume(reject) is not None, "| second:", engine.resume(reject))
runner.run_until_idle()
r = store.get(run.run_id); show(r); show_journal(r)
assert pay.charges == {} and r.result == {"result": "Not paid: vendor not onboarded."}

## Swap the model: Mistral decides, by function calling

`lra.adapters.mistral.MistralDecider` turns the journal into a chat — the goal, then every earlier decision as an assistant tool call with its recorded result — and asks for the next call with `tool_choice="auto"`. The fake client below speaks the SDK's response shape, so this runs offline; with `pip install -e ".[mistral]"` and `MISTRAL_API_KEY`, `MistralDecider(TOOL_SPECS)` calls the real API (`python -m lra.adapters.mistral.demo live`). Mistral requires tool-call ids of exactly nine alphanumeric characters, so the adapter hashes the journal key.

In [ ]:
from types import SimpleNamespace as NS
from lra.adapters.mistral import DEFAULT_INSTRUCTIONS, TOOL_SPECS, MistralDecider, MistralLLM, to_messages, tool_call_id

class FakeMistral:
    """Scripted chat completions in the SDK's response shape; keeps every request."""
    def __init__(self, script):
        self.script, self.requests, self.chat = list(script), [], self
    def complete(self, **kw):
        self.requests.append(kw)
        item = self.script.pop(0)
        if "tool" in item:
            msg = NS(content="", tool_calls=[NS(id="abc123def", type="function",
                                                  function=NS(name=item["tool"], arguments=json.dumps(item["args"])))])
        else:
            msg = NS(content=item["final"], tool_calls=None)
        return NS(choices=[NS(message=msg)], usage=NS(prompt_tokens=900, completion_tokens=40))

fake = FakeMistral([{"tool": "charge", "args": {"amount": 42}}, {"final": "Charged 42."}])
pay = PaymentAPI()
engine, runner, clock, store = tool_harness(make_tool_agent(MistralDecider(TOOL_SPECS, model="mistral-small-latest", client=fake), {"charge": pay}))
run = engine.start("tool_agent", {"goal": "Pay supplier invoice INV-1042: 42 SGD."})
runner.run_until_idle()
r = store.get(run.run_id)
engine.resume(Event(run_id=run.run_id, key=r.wait.key, payload={"decision": "approve", "by": "cfo"}))
runner.run_until_idle()
r = store.get(run.run_id); show(r)
print("second prompt:", [m["role"] for m in fake.requests[1]["messages"]])
print("the tool result the model saw:", fake.requests[1]["messages"][-1]["content"])
assert r.status == RunStatus.SUCCEEDED and pay.charges == {f"{run.run_id}:1": 42.0}

## Your turn: the journal as the prompt

Rebuild `to_messages`: each decision becomes an assistant tool call, and the intent journaled right after it becomes the matching tool message — its recorded result, or `{"pending": true}` if it has not run. Fill in the tool-call id and the tool message's content.

In [ ]:
# TODO: fill in the blanks: cid, content
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in the blanks: cid, content")
def my_to_messages(goal, journal):
    messages = [{"role": "system", "content": DEFAULT_INSTRUCTIONS}, {"role": "user", "content": goal}]
    for i, step in enumerate(journal):
        if step["type"] != "decision" or "final" in step:
            continue
        intent = journal[i + 1]
        cid = ____
        messages.append({"role": "assistant", "content": "", "tool_calls": [
            {"id": cid, "type": "function", "function": {"name": step["tool"], "arguments": json.dumps(step["args"])}}]})
        messages.append({"role": "tool", "tool_call_id": cid, "name": step["tool"], "content": json.dumps(____)})
    return messages

journals = [
    [],
    r.state["journal"],
    [{"type": "decision", "tool": "charge", "args": {"amount": 1}},
     {"type": "intent", "tool": "charge", "args": {"amount": 1}, "key": "r:1", "done": False}],
    [{"type": "decision", "tool": "charge", "args": {"amount": 5}},
     {"type": "intent", "tool": "charge", "args": {"amount": 5}, "key": "r:1", "done": True, "result": {"rejected": "no"}},
     {"type": "decision", "final": "stopped"}],
]
for j in journals:
    assert my_to_messages("goal", j) == to_messages("goal", j), j
print("my_to_messages matches the adapter on", len(journals), "journals; ids look like", tool_call_id("r:1"))

## Mistral as the engine's model

`MistralLLM` implements the same `LLM` port as `GeminiLLM` and `FakeLLM`, so every workflow in `lra.examples` runs on it unchanged; usage becomes cost for the run's budget (the prices in the adapter are illustrative).

In [ ]:
fake = FakeMistral([{"final": "billing"}])
wf = Workflow("classify")
@wf.step(start=True)
def classify(ctx):
    return Done(ctx.llm("Classify this ticket as billing/technical/other: I was charged twice").text)
clock, store = FakeClock(), InMemoryStateStore(); queue = InMemoryTaskQueue(clock)
engine = Engine(store=store, queue=queue, bus=InMemoryEventBus(), llm=MistralLLM(model="mistral-small-latest", client=fake),
                clock=clock, workflows=[wf])
run = engine.start("classify", {}); LocalRunner(engine, queue, clock).run_until_idle()
r = store.get(run.run_id)
print(r.status.value, r.result, "| tokens:", r.budget.tokens_used, "| cost (illustrative prices): $%.6f" % r.budget.cost_usd)
assert r.result == "billing" and r.budget.tokens_used == 940

## The same loop on Mistral Workflows

`lra/adapters/mistral/workflow.py` is this loop on Mistral Workflows, with Temporal underneath. What disappears is the plumbing: no store, no queue, no lease, no reaper. The execution's **event history** is the store; every **activity** (the model call and the charge) is recorded before it runs and retried on failure, never re-run once complete; `workflow.wait_condition(...)` plus a **signal** is the approval gate, with a timeout; a loop bound in deterministic workflow code plus `execution_timeout` is the budget. One subtlety: an *unexpected* exception in workflow code fails only the workflow **task**, which the platform retries until the code is fixed and redeployed; to fail an execution on purpose, raise `WorkflowError`.

It needs Python 3.12–3.14 and the `mistral` extra (`mistralai-workflows` 3.15 declares `Requires-Python >=3.12,<3.15`; checked 2026-09-26, verify), and a local Temporal dev server that the SDK downloads on first use. `python -m lra.adapters.mistral.demo workflow` runs the approval, crash-after-charge and timeout scenarios; `tests/test_mistral_workflow.py` checks them. The cell below shows the gate when the extra is installed.

In [ ]:
import importlib.util, inspect, sys
if sys.version_info >= (3, 12) and importlib.util.find_spec("mistralai") and importlib.util.find_spec("mistralai.workflows"):
    from lra.adapters.mistral import workflow as mw
    print(inspect.getsource(mw.InvoiceAgent.run))
else:
    print("Mistral Workflows is not installed here (Python 3.12+ and the mistral extra); read lra/adapters/mistral/workflow.py.")

## Takeaways

* A model-chosen action is a side effect twice over: record the *decision*, then the *act*, each under its own key.
* The journal is the prompt on every retry: recorded facts, never the model's memory.
* The provider is one adapter (`MistralDecider`, `MistralLLM`); the invariants do not move. A durable-execution platform (Mistral Workflows, Temporal, Cloud Workflows) provides them for you, and you still pass idempotency keys downstream.